<a href="https://colab.research.google.com/github/tinadams/carisurg-portfolio/blob/feat%2Fweek-0-refactor/Day1_CleanGenderColumn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 0 Day 1: Cleaning the Gender Column

**Project:** Mercer General Hospital Emergency Department Triage Dataset  
**Author:** Tessa Adams  
**Date:** 20 May 2026  

## Objective

This notebook cleans the `Gender` column in the Mercer General Emergency Department dataset. The goal is to standardise inconsistent gender entries so the column can be used in later triage modelling work.

## Cleaning Rules

The `Gender` column is mapped as follows:

- `1` = Male
- `0` = Female
- `-1` = Unknown or missing

## 1. Mount Google Drive

This step connects Google Colab to Google Drive so the dataset can be accessed if it is stored there.

If the CSV file is uploaded directly into the Colab session instead, this step may not be required.

In [11]:
from google.colab import drive
drive.mount ('/content/drive')
print ("Drive mounted successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully!


## 2. Confirm Python Environment

This step checks that the Python version used in the notebook is compatible.

In [12]:
# Always confirm Python version first
import sys
print(f"Python version: {sys.version}")


Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## Import Libraries and Load Dataset

This step imports `pandas`, the main Python library used for working with tabular data, and loads the synthetic Mercer General Emergency Department dataset into a dataframe called `df_raw`.

In [13]:
# Import pandas
import pandas as pd

# Import the dataset
df_raw = pd.read_csv("EmergencyTriageDataset_Reduced_Dirty.csv")

## Inspect Original Gender Values

This step checks the original values in the `Gender` column before cleaning. Reviewing the unique values and counts helps identify inconsistent labels that need to be standardised.

In [14]:
# What values are actually in the Gender column?
print("Unique values in Gender:")
print(df_raw['Gender'].unique())
print(f"\nTotal unique values: {df_raw['Gender'].nunique()}")

# Count how many of each value we have
print("\nGender value counts:")
print(df_raw['Gender'].value_counts())

Unique values in Gender:
['0' 'Male' 'Female' 'FEMALE' '1' 'MALE']

Total unique values: 6

Gender value counts:
Gender
1         422
MALE      379
Male      375
FEMALE    366
Female    340
0         323
Name: count, dtype: int64


## Standardise Gender Text Values

This step normalises the `Gender` column by converting values to text, removing extra spaces, and changing all entries to lowercase. This makes the values easier to map consistently.

In [15]:
# Normalise the Gender column
df_raw['Gender'] = (
    df_raw['Gender']
    .astype(str)
    .str.strip()
    .str.lower()
)

# What values are actually in the Gender column?
print("\nUnique values in Gender after normalization:")
print(df_raw['Gender'].unique())
print(f"\nTotal unique values after normalization: {df_raw['Gender'].nunique()}")


Unique values in Gender after normalization:
['0' 'male' 'female' '1']

Total unique values after normalization: 4


## Map Gender Values to Numeric Labels

This step handles blank or missing `Gender` values, then maps the cleaned text values into numeric labels for analysis.

The mapping used is:

- `1` = Male
- `0` = Female
- `2` = Non-binary or other
- `-1` = Unknown

After applying the mapping, the notebook checks whether any unmapped values remain.

In [16]:
# Handle missing or blank values (NaN values)
df_raw['Gender'] = df_raw['Gender'].replace(['', 'nan', 'unknown'], 'unknown')


# Build mapping (now matching normalised values)
gender_map = {
    'male': 1,
    'm': 1,
    '1': 1,

    'female': 0,
    'f': 0,
    '0': 0,

    'non-binary': 2,
    'nonbinary': 2,
    'nb': 2,
    'other': 2,

    'unknown': -1
}

# Apply the mapping to create a new, clean column
df_raw['Gender_Clean'] = df_raw['Gender'].map(gender_map)


# Check for NaN values
print(f"\nAny NaN values? {df_raw['Gender_Clean'].isnull().sum()}")

# Check for non-binary or other values
print("\nNon-binary/other entries:")
print(df_raw[df_raw['Gender'].isin(['non-binary', 'nonbinary', 'nb', 'other'])])




Any NaN values? 0

Non-binary/other entries:
Empty DataFrame
Columns: [ID, Age, Gender, GCS, SBP, DBP, MAP, pulse, Temp, RR, Fio2, Gender_Clean]
Index: []


## Finalise Cleaned Gender Column

This step replaces the original `Gender` column with the cleaned version and checks the final output.

In [17]:
# Drop the original dirty column now that we have a clean one
df_raw = df_raw.drop(columns=['Gender'])
df_raw = df_raw.rename(columns={'Gender_Clean': 'Gender'})

# Confirm the result
print("\nColumn 'Gender' after cleaning \n(where 0 = female, 1 = male,"\
      " 2 = non-binary, -1 = unknown):\n")
print(df_raw['Gender'].value_counts())
print(f"Data type: {df_raw['Gender'].dtype}")
print(f"NaNs: {df_raw['Gender'].isnull().sum()}")
print("\n")
print(df_raw.head())


Column 'Gender' after cleaning 
(where 0 = female, 1 = male, 2 = non-binary, -1 = unknown):

Gender
1    1176
0    1029
Name: count, dtype: int64
Data type: int64
NaNs: 0


   ID  Age   GCS  SBP    DBP     MAP  pulse  Temp    RR   Fio2  Gender
0   1   34  15.0   93   67.0   75.67  128.0  36.8  14.0   21.0       0
1   2   20  15.0  130   90.0  103.33   80.0  37.0  16.0   21.0       1
2   3   77  14.0  163  105.0  124.33   92.0  36.8  18.0   21.0       0
3   4   23   8.0  100   60.0   73.33  100.0  37.0  12.0  100.0       0
4   5   86  15.0  150   90.0  110.00   85.0  37.0  19.0   21.0       0
